**Load the Dataset**

In [8]:
from google.colab import drive
drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/Colab Notebooks/20_ML_Projects/Hotel_Reviews.csv'

Mounted at /content/drive


**Import Libraries**

In [22]:
import nltk
nltk.download('wordnet')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('mw-1.4')
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.wordnet import WordNetLemmatizer
from ast import literal_eval


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Error loading mw-1.4: Package 'mw-1.4' not found in index


In [10]:
data = pd.read_csv(file_path)

In [11]:
data.head()

,Hotel_Address,Additional_Number_of_Scoring,Review_Date,Average_Score,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Reviewer_Score,Tags,days_since_review,lat,lng
0,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,8/3/2017,7.7,Hotel Arena,Russia,I am so angry that i made this post available...,397,1403,Only the park outside of the hotel was beauti...,11,7,2.9,"[' Leisure trip ', ' Couple ', ' Duplex Double...",0 days,52.360576,4.915968
1,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,8/3/2017,7.7,Hotel Arena,Ireland,No Negative,0,1403,No real complaints the hotel was great great ...,105,7,7.5,"[' Leisure trip ', ' Couple ', ' Duplex Double...",0 days,52.360576,4.915968
2,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/31/2017,7.7,Hotel Arena,Australia,Rooms are nice but for elderly a bit difficul...,42,1403,Location was good and staff were ok It is cut...,21,9,7.1,"[' Leisure trip ', ' Family with young childre...",3 days,52.360576,4.915968
3,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/31/2017,7.7,Hotel Arena,United Kingdom,My room was dirty and I was afraid to walk ba...,210,1403,Great location in nice surroundings the bar a...,26,1,3.8,"[' Leisure trip ', ' Solo traveler ', ' Duplex...",3 days,52.360576,4.915968
4,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/24/2017,7.7,Hotel Arena,New Zealand,You When I booked with your company on line y...,140,1403,Amazing location and building Romantic setting,8,3,6.7,"[' Leisure trip ', ' Couple ', ' Suite ', ' St...",10 days,52.360576,4.915968


In [ ]:
data.shape

(515738, 17)

In [12]:
data.Hotel_Address[67]

' s Gravesandestraat 55 Oost 1092 AA Amsterdam Netherlands'

In [13]:
data.Hotel_Address = data.Hotel_Address.str.replace('United Kingdom', 'UK')
data['Countries'] = data.Hotel_Address.apply(lambda x: x.split(' ')[-1])      # New column is created which contains the name of countries
print(data.Countries.unique())

['Netherlands' 'UK' 'France' 'Spain' 'Italy' 'Austria']


In [ ]:
data.Countries.value_counts()

,count
Countries,
UK,262301
Spain,60149
France,59928
Netherlands,57214
Austria,38939
Italy,37207


In [14]:
data.columns

Index(['Hotel_Address', 'Additional_Number_of_Scoring', 'Review_Date',
       'Average_Score', 'Hotel_Name', 'Reviewer_Nationality',
       'Negative_Review', 'Review_Total_Negative_Word_Counts',
       'Total_Number_of_Reviews', 'Positive_Review',
       'Review_Total_Positive_Word_Counts',
       'Total_Number_of_Reviews_Reviewer_Has_Given', 'Reviewer_Score', 'Tags',
       'days_since_review', 'lat', 'lng', 'Countries'],
      dtype='object')

**Data Cleaning**

In [15]:
# Dropping Unecessary Columns
data.drop(columns=['Additional_Number_of_Scoring','Review_Date',
       'Reviewer_Nationality','Negative_Review', 'Review_Total_Negative_Word_Counts',
       'Total_Number_of_Reviews', 'Positive_Review',
       'Review_Total_Positive_Word_Counts',
       'Total_Number_of_Reviews_Reviewer_Has_Given', 'Reviewer_Score',
       'days_since_review', 'lat', 'lng'],inplace=True)

In [16]:
def impute(column):
  column = column[0]
  if (type(column) != list):
    return ''.join(literal_eval(column))      # Tags are stored in strings that look like lists, so we are converting them into clean strings
  else:
    return column

data['Tags'] = data[['Tags']].apply(impute, axis=1)
data.head()

/tmp/ipykernel_1182/4108964394.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  column = column[0]


,Hotel_Address,Average_Score,Hotel_Name,Tags,Countries
0,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,7.7,Hotel Arena,Leisure trip Couple Duplex Double Room Sta...,Netherlands
1,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,7.7,Hotel Arena,Leisure trip Couple Duplex Double Room Sta...,Netherlands
2,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,7.7,Hotel Arena,Leisure trip Family with young children Dup...,Netherlands
3,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,7.7,Hotel Arena,Leisure trip Solo traveler Duplex Double Ro...,Netherlands
4,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,7.7,Hotel Arena,Leisure trip Couple Suite Stayed 2 nights ...,Netherlands


In [17]:
data['Countries'] = data['Countries'].str.lower()
data['Tags'] = data['Tags'].str.lower()

**The Main Function**

In [28]:
def recommend_hotel(location, description):                # A Function that takes user's location and hotel requirements
  description = description.lower()
  description = word_tokenize(description)                 # Splitting description into individual words (tokens)
  stop_words = set(stopwords.words('english'))             # Loading english stop words into a set for faster lookup
  lemm = WordNetLemmatizer()                               # Creating a lemmatizer object

  # Remove Stop words and convert words into thier base form
  user_words = {
      lemm.lemmatize(word)
      for word in description
      if word.isalpha() and word not in stop_words
  }
  # Filter hotels belonging to the selected country
  country = data[data['Countries'] == location.lower()].copy()

  # If no hotels are found, return a message
  if country.empty:
   return 'No Hotels found for this location'

  similarity_scores = []

   # Loop through every hotel in the selected country
  for tags in country['Tags']:
    hotel_tokens = word_tokenize(tags.lower())         # Spliting hotel tags into words

   # Remove stopwords and lemmatize hotel tags
    hotel_words = {
       lemm.lemmatize(word)
       for word in hotel_tokens
       if word.isalpha() and word not in stop_words
    }
    similarity = len(user_words.intersection(hotel_words))     # Counting the no. of common words
    similarity_scores.append(similarity)                       # Saving similarity scores

  country['Similarity'] = similarity_scores                  # Add similariy scores as a new column

   # Sorting hotels by similarity first, then by average rating
  country = country.sort_values(by=['Similarity', 'Average_Score'],
                                 ascending=[False, False])

   # Removing duplicate hotel names
  country = country.drop_duplicates(subset='Hotel_Name')

   # Retruning top 5 recommended hotels
  return country[[
       'Hotel_Name',
       'Average_Score',
       'Hotel_Address',
       'Similarity'
   ]].head(5)


**Get your recommendations!!**



In [32]:
recommend_hotel('Spain','I am going for a business trip')

,Hotel_Name,Average_Score,Hotel_Address,Similarity
316456,Hotel Casa Camper,9.6,Elisabets 11 Ciutat Vella 08001 Barcelona Spain,2
399033,Hotel The Serras,9.6,Passeig de Colom 9 Ciutat Vella 08002 Barcelon...,2
402252,H10 Casa Mimosa 4 Sup,9.6,Pau Claris 179 Eixample 08037 Barcelona Spain,2
312810,Mercer Hotel Barcelona,9.5,Dels Lledo 7 Ciutat Vella 08003 Barcelona Spain,2
129320,The One Barcelona GL,9.4,277 Carrer de Proven a Eixample 08037 Barcelon...,2


In [30]:
recommend_hotel('UK','I am going on a honeymoon, i need honeymoon suite room for 3 days')

,Hotel_Name,Average_Score,Hotel_Address,Similarity
223863,Club Quarters Hotel Lincoln s Inn Fields,8.9,61 Lincoln s Inn Fields Camden London WC2A 3JW UK,2
248358,Club Quarters Hotel Trafalgar Square,8.5,8 Northumberland Avenue Westminster Borough Lo...,2
119278,Club Quarters Hotel St Paul s,8.4,24 Ludgate Hill City of London London EC4M 7DR UK,2
143325,Rathbone,8.3,30 Rathbone Street West End Westminster Boroug...,2
103905,The Academy,7.8,21 Gower Street Camden London WC1E 6HG UK,2


# **Hotel Recommendation System Using NLP**
# **Project Summary**

This project builds a **Hotel Recommendation System** using **Natural Language Processing (NLP)** techniques. The system recommends hotels based on the user's preferred **location** and a **textual description** of their travel requirements. Instead of using ratings alone, the recommendation is generated by comparing the user's preferences with descriptive hotel tags available in the dataset.

## **Dataset Overview**

| Feature | Description |
|----------|-------------|
| Hotel_Name | Name of the hotel |
| Hotel_Address | Complete address of the hotel |
| Countries | Country extracted from the hotel address |
| Tags | Keywords describing the hotel's facilities and characteristics |
| Average_Score | Average customer rating of the hotel |

## **Project Workflow**

The project begins by loading the hotel reviews dataset and performing data preprocessing. The hotel addresses are cleaned by replacing lengthy country names with shorter forms, and a new **Countries** column is created by extracting the country name from each hotel's address. Several unnecessary columns related to reviews, reviewer information, dates, and geographical coordinates are removed to simplify the dataset.

The **Tags** column, which stores hotel characteristics, is cleaned and converted into a readable format. Both the country names and hotel tags are converted to lowercase to ensure consistent text matching during the recommendation process.

A custom recommendation function is then created that accepts two inputs: the **country** where the user wants to stay and a **text description** of their requirements. The user description is processed using **Natural Language Processing (NLP)** techniques. The text is tokenized into individual words, common English stop words are removed, and the remaining words are lemmatized to convert them into their root forms.

The system filters all hotels belonging to the selected country and compares the processed user keywords with each hotel's tags. A **similarity score** is calculated by counting the number of common words between the user's preferences and the hotel's descriptive tags. Hotels are then ranked primarily by similarity score and secondarily by their average customer rating. Duplicate hotel names are removed before returning the **Top 5 most relevant hotel recommendations**.

## NLP Techniques Used

- **Tokenization** – Splits text into individual words.
- **Stop Word Removal** – Removes common words such as *the*, *is*, and *and* that do not contribute to meaning.
- **Lemmatization** – Converts words to their base form (e.g., *travelling* → *travel*, *rooms* → *room*).
- **Keyword Matching** – Measures similarity by counting common words between user preferences and hotel tags.

## Recommendation Example

For the query:

- **Location:** Spain
- **Description:** *"I am going for a business trip"*

the system analyzes keywords such as **business** and **trip**, compares them with the hotel tags of properties located in Spain, calculates similarity scores, and recommends the **Top 5 hotels** that best match the user's travel requirements while also considering their average ratings.

## **Conclusion**

This project demonstrates how **Natural Language Processing (NLP)** can be combined with a simple **content-based recommendation approach** to build an intelligent hotel recommendation system. By understanding the user's textual preferences instead of relying solely on ratings, the system provides personalized hotel suggestions based on both location and hotel characteristics.